# 02. Data Cleaning & Feature Engineering

## 1. Project Setup

### 1.1 Import Libraries


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

### 1.2 Environment Check


In [2]:
print("Numpy version: ", np.__version__)
print("Pandas version: ", pd.__version__)

Numpy version:  2.5.4
Pandas version:  3.1.1


### 1.3 Set Project Path


In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Sample-Superstore.csv"

print("Project Root: ", PROJECT_ROOT)
print("Dataset exists: ", DATA_PATH.exists())
print("Dataset path: ", DATA_PATH)

Project Root:  c:\Users\pc\Desktop\Retail-Sales-and-Business-Analysis
Dataset exists:  True
Dataset path:  c:\Users\pc\Desktop\Retail-Sales-and-Business-Analysis\data\raw\Sample-Superstore.csv


### 1.4 Load Raw Dataset


In [4]:
raw_df = pd.read_csv(DATA_PATH)
print(f"Raw dataset shape: {raw_df.shape}")

Raw dataset shape: (10194, 21)


## 2. Working Dataset Preparation

### 2.1 Create Working Dataset


Creating a separate working copy of the raw dataset so the original data stays untouched. All cleaning will be done on the working copy.

In [5]:
clean_df = raw_df.copy()

print("Raw shape:", raw_df.shape)
print("Working copy shape:", clean_df.shape)
print("Same object:", clean_df is raw_df)

Raw shape: (10194, 21)
Working copy shape: (10194, 21)
Same object: False


### 2.2 Convert Date Columns to Datetime


In [6]:
clean_df['Order Date'] = pd.to_datetime(clean_df['Order Date'])
clean_df['Ship Date'] = pd.to_datetime(clean_df['Ship Date'])

## 3. Future-Date Investigation
Objective: Identify order records with dates later than November 15, 2026, and calculate their count and percentage of the complete dataset.


#### 3.1.1 Future Order Date Count & Percentage

1. Identify Future-Dated Order Records

In [7]:
reference_date = pd.Timestamp('2026-11-15')
future_orders = clean_df[clean_df['Order Date'] > reference_date]

2. Calculate Total Records and Percentage

In [8]:
future_orders_count = len(future_orders)
print("Future order records:", future_orders_count)

future_order_pct = future_orders_count / len(clean_df) * 100
print(f"Future order percentage: {future_order_pct:.2f}%")

Future order records: 709
Future order percentage: 6.96%


Observations:

Using November 15, 2026, as the reference date, the investigation identified 709 records with Order Dates later than the reference date.

These records represent 6.96% of the complete dataset.

This indicates that a portion of the dataset contains future-dated order records, which should be investigated further before using the data for business analysis.

#### 3.1.2 Future Order Date Range & Distribution
Objective: Check the earliest/latest dates, total span in days, sample records, and monthly distribution of future-dated orders.

1. Earliest, Latest and Date Span

In [9]:
earliest_future_order = future_orders['Order Date'].min()
latest_future_order = future_orders['Order Date'].max()
print("Earliest future order date:", earliest_future_order)
print("Latest future order date:",latest_future_order)

future_order_span_days = (latest_future_order - earliest_future_order).days
print("Future orders span (days):", future_order_span_days)

Earliest future order date: 2026-11-16 00:00:00
Latest future order date: 2026-12-30 00:00:00
Future orders span (days): 44


2. Sample Future-Dated Order Records

In [10]:
future_orders_sample = future_orders.sample(5, random_state=42)
print(future_orders_sample[['Order ID','Order Date','Sales']])

future_order_by_month = future_orders['Order Date'].dt.month.value_counts().sort_index()
print("\nFuture order by month:")
print(future_order_by_month)

             Order ID Order Date    Sales
9779   US-2026-167395 2026-12-02  979.950
9741   US-2026-144862 2026-12-01   62.958
10098  US-2026-117443 2026-12-23   13.840
9842   US-2026-149006 2026-12-06   10.680
9733   US-2026-118542 2026-12-01   45.360

Future order by month:
Order Date
11    227
12    482
Name: count, dtype: int64


Observations:

- The earliest future Order Date is November 16, 2026, and the latest is December 30, 2026, a span of 44 days.
- November has 227 future-dated records (about 15 per day over 15 days) and December has 482 (about 16 per day over 30 days), a similar daily rate. A daily distribution check was not performed.
- The records form a continuous stretch, not a few isolated entries, which suggests the dates are part of the dataset's overall date range.

### 3.2 Future Ship Date Investigation

Objective: Identify shipment records with dates later than November 15, 2026, and calculate their count and percentage of the complete dataset.

#### 3.2.1 Future Ship Date Count & Percentage

1. Identify Future-Dated Shipments

In [11]:
future_ships = clean_df[clean_df['Ship Date'] > reference_date]

2. Calculate Total Records and Percentage

In [12]:
future_ship_count = len(future_ships)
print("Future ship records:", future_ship_count)

future_ship_pct = future_ship_count / len(clean_df) * 100
print(f"Future ship percentage: {future_ship_pct:.2f}%")

Future ship records: 775
Future ship percentage: 7.60%


#### 3.2.2 Orders Placed Before but Shipped After Reference Date

In [13]:
shipped_after = clean_df[(clean_df['Order Date'] <= reference_date) & (clean_df['Ship Date'] > reference_date)]
print("Order before, ship after:", len(shipped_after))

print(shipped_after[['Order ID', 'Order Date', 'Ship Date']].head())

Order before, ship after: 66
            Order ID Order Date  Ship Date
9390  US-2026-137624 2026-11-10 2026-11-17
9393  US-2026-128447 2026-11-10 2026-11-17
9397  US-2026-128447 2026-11-10 2026-11-17
9398  US-2026-104640 2026-11-10 2026-11-16
9401  US-2026-141138 2026-11-10 2026-11-16


**Observations:**

- The investigation identified 775 records with Ship Dates later than November 15, 2026, which is 7.60% of the complete dataset.
- Future-dated shipments (775) exceed future-dated orders (709) by 66 records.
- A separate check confirmed that exactly 66 records have an Order Date on or before November 15, 2026, but a Ship Date after it.
- The sampled records were ordered on November 10, 2026, and shipped on November 16 or 17, 2026. This matches the normal shipping time of a few days, so the extra 66 records are orders placed shortly before the reference date whose shipments fall after it.
- Every future-dated order also ships after the reference date, so the 775 future shipments are the 709 future-dated orders plus these 66 records.

These findings show that future-dated shipment records come from the same date range as the future-dated orders, not from separate or unusual entries. The decision on how to handle them will be made at the end of Mission 2.1.

#### 3.2.3 Future Ship Date Range & Distribution

Objective: Investigate the date range and distribution of future-dated Ship Dates to understand how far they extend and whether they are spread across the period or concentrated on specific dates.

1. Earliest, Latest and Date Span

In [14]:
earliest_future_ship = future_ships['Ship Date'].min()
latest_future_ship = future_ships['Ship Date'].max()
print("Earliest future ship date:", earliest_future_ship)
print("Latest future ship date:",latest_future_ship)

future_ship_span_days = (latest_future_ship - earliest_future_ship).days
print("Future ship span (days):", future_ship_span_days)

Earliest future ship date: 2026-11-16 00:00:00
Latest future ship date: 2027-01-05 00:00:00
Future ship span (days): 50


2. Sample Future-Dated ship Records

In [15]:
future_ships_sample = future_ships.sample(5, random_state=42)
print(future_ships_sample[
    ['Order ID', 'Order Date', 'Ship Date', 'Sales']
])

future_ship_by_month = future_ships['Ship Date'].dt.month.value_counts().sort_index()
print("\nFuture ship by month:")
print(future_ship_by_month)

             Order ID Order Date  Ship Date    Sales
9847   US-2026-134285 2026-12-07 2026-12-12   15.552
10131  US-2026-134194 2026-12-25 2027-01-01   40.980
9745   US-2026-145233 2026-12-01 2026-12-05  470.376
10184  CA-2026-141531 2026-12-29 2027-01-02   68.460
9780   CA-2026-105471 2026-12-02 2026-12-06   58.360

Future ship by month:
Ship Date
1      46
11    244
12    485
Name: count, dtype: int64


Observations:

- The earliest future Ship Date is November 16, 2026, and the latest is January 5, 2027, a span of 50 days.
- Future Ship Dates fall in three months: November (244 records), December (485 records) and January 2027 (46 records).
- November covers 15 days (about 16 records per day) and December covers 31 days (about 16 per day). January covers only 5 days (about 9 per day), and these are the shipments of orders placed at the very end of December.
- The sampled records were ordered in December 2026 and ship within the same future window.

Decision / Conclusion:

Retain all future-dated records. No Ship Date falls before its Order Date (Day 1), and the future-dated records form a continuous stretch of the dataset's date range, not isolated entries. Being later than the reference date does not make a record invalid, so no records are deleted or shifted.

## 4. Data Cleaning

Objective: Decide which cleaning actions are necessary, based on the Day 1 investigation

### 4.1 Cleaning Decisions


Decisions:

| Issue | Day 1 Finding | Decision |
|---|---|---|
| Missing values | 0 in all 21 columns | No treatment required |
| Exact duplicates | 0 records | No removal required |
| Repeated Order IDs | Multiple products within one order | Retain |
| Data types | Dates converted to datetime in setup | No further change |
| Quantity | 1 to 14, no invalid values | Retain |
| Discount | 0.00 to 0.80, within valid range | Retain |
| Negative profit | 1,901 loss-making order-line records | Retain, not an error |
| Statistical outliers | IQR outliers in Sales, Quantity, Discount | Retain, not errors |
| Future-dated records | 709 orders, 775 ship dates beyond Nov 15, 2026 | Retain, part of the dataset's date range |


### 4.2 Data Cleaning Conclusion

Based on the Day 1 investigation, no unnecessary cleaning operations were required.

- No missing values or exact duplicate records were found.
- Repeated Order IDs represent multiple order lines and were retained.
- Negative profit values represent genuine loss-making records.
- Statistical outliers were retained because they are not necessarily errors.
- Quantity and discount values fall within the observed valid ranges.
- Future-dated records were retained.

No records were deleted or modified during this cleaning investigation.

The dataset is ready to proceed to feature engineering, with future-dated records documented for further review.

## 5. Feature Engineering


Objective: Create new analytical features from existing columns to support the business analysis in the upcoming days.

### 5.1 Date-Based Features
Objective: Create Order Year, Order Month, Year-Month, and Shipping Days columns from Order Date and Ship Date for yearly, monthly, and shipping performance analysis.

Business Purpose:

Separate year, month, and Year-Month columns make yearly and monthly trend analysis easier. A Shipping Days column makes it possible to analyze shipping performance.

#### 5.1.1 Create Date-Based Features


In [16]:
clean_df["Order Year"] = clean_df['Order Date'].dt.year
clean_df['Order Month'] = clean_df['Order Date'].dt.month
clean_df['Year-Month'] = clean_df['Order Date'].dt.strftime('%Y-%m')

#### 5.1.2 Verify Date-Based Features

In [17]:
print(clean_df[['Order Year', 'Order Month', 'Year-Month']].head())

print("\nRecords per year:")
print(clean_df['Order Year'].value_counts().sort_index())

print("\nNew columns data types:")
print(clean_df[['Order Year', 'Order Month', 'Year-Month']].dtypes)

   Order Year  Order Month Year-Month
0        2023            1    2023-01
1        2023            1    2023-01
2        2023            1    2023-01
3        2023            1    2023-01
4        2023            1    2023-01

Records per year:
Order Year
2023    2051
2024    2130
2025    2634
2026    3379
Name: count, dtype: int64

New columns data types:
Order Year     int32
Order Month    int32
Year-Month       str
dtype: object


Observations:

- Three new columns were created: Order Year, Order Month (1 to 12, integer) and Year-Month (text such as "2023-01").
- Records per year: 2023 has 2,051, 2024 has 2,130, 2025 has 2,634 and 2026 has 3,379. The total is 10,194, matching the dataset size.
- The number of order records grows every year. The 2026 count includes the 709 future-dated records identified earlier, which were retained.

Decision / Conclusion:

The date-based features were created correctly and added to the working dataset (clean_df). The raw dataset is unchanged.

#### 5.1.3 Create Shipping Days

In [18]:
clean_df['Shipping Days'] = (clean_df['Ship Date'] - clean_df['Order Date']).dt.days

#### 5.1.4 Verify Shipping Days

In [19]:
print(clean_df[['Order Date', 'Ship Date', 'Shipping Days']].head())
print("Min / Max / Average:", clean_df['Shipping Days'].min(), "/", clean_df['Shipping Days'].max(), "/", round(clean_df['Shipping Days'].mean(), 2))

  Order Date  Ship Date  Shipping Days
0 2023-01-03 2023-01-07              4
1 2023-01-04 2023-01-08              4
2 2023-01-04 2023-01-08              4
3 2023-01-04 2023-01-08              4
4 2023-01-05 2023-01-12              7
Min / Max / Average: 0 / 11 / 3.96


Observations:

The Shipping Days column was created from Ship Date minus Order Date. Its minimum (0), maximum (11) and average (3.96) match the Day 1 shipping duration results.

Decision / Conclusion:

The Shipping Days feature was created correctly and added to the working dataset.

### 5.2 Profit-Based Feature
Objective: Create a Profit Status column to classify each record as Profitable, Loss, or Break-even.

#### 5.2.1 Create Profit Status

In [20]:
is_profitable = clean_df['Profit'] > 0
is_loss = clean_df['Profit'] < 0
is_breakeven = clean_df['Profit'] == 0

conditions = [is_profitable, is_loss, is_breakeven]
choices = ['Profitable', 'Loss', 'Break-even']

clean_df['Profit Status'] = np.select(conditions, choices, default='Unknown')

#### 5.2.2 Verify Profit Status


In [21]:
print(clean_df['Profit Status'].value_counts())
print("\nTotal:", clean_df['Profit Status'].value_counts().sum())

print("\nSample check:")
print(clean_df[['Profit', 'Profit Status']].sample(5, random_state=42))

Profit Status
Profitable    8228
Loss          1901
Break-even      65
Name: count, dtype: int64

Total: 10194

Sample check:
         Profit Profit Status
6636   -14.3313          Loss
3742     4.7712    Profitable
5114    -7.7094          Loss
6505   108.1752    Profitable
10072  218.2518    Profitable


Observations:

The Profit Status column was created with three categories. Of the 10,194 records, 8,228 are Profitable, 1,901 are Loss, and 65 are Break-even. The categories add up to the full dataset, no record was left unclassified, and the Loss count matches the Day 1 result. The sampled records also show the correct labels for positive and negative profit values.

Decision / Conclusion:

The Profit Status feature was created correctly and added to the working dataset (clean_df). The raw dataset is unchanged.

## 6. Final Validation & Saving
Objective: Validate the working dataset and save the processed dataset for the business analysis in the upcoming days.

### 6.1 Final Validation


Objective: Verify the shape, new features, missing values, duplicates, and data types of the working dataset, and confirm that the raw dataset is unchanged.

In [22]:
new_features = ['Order Year', 'Order Month', 'Year-Month', 'Shipping Days', 'Profit Status']
print("Clean shape:", clean_df.shape)
print("All 5 new features present:", all(col in clean_df.columns for col in new_features))
print("Missing values:", clean_df.isnull().sum().sum())
print("Exact duplicates:", clean_df.duplicated().sum())

print("\nRaw shape:", raw_df.shape)
print("New columns in raw_df:", any(col in raw_df.columns for col in new_features))

print("\nData types:")
print(clean_df.dtypes)

Clean shape: (10194, 26)
All 5 new features present: True
Missing values: 0
Exact duplicates: 0

Raw shape: (10194, 21)
New columns in raw_df: False

Data types:
Row ID                     int64
Order ID                     str
Order Date        datetime64[us]
Ship Date         datetime64[us]
Ship Mode                    str
Customer ID                  str
Customer Name                str
Segment                      str
Country/Region               str
City                         str
State/Province               str
Postal Code                  str
Region                       str
Product ID                   str
Category                     str
Sub-Category                 str
Product Name                 str
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
Order Year                 int32
Order Month                int32
Year-Month                   str
Shipping Days              int64
Profit Status

Observations:

The working dataset has 10,194 rows and 26 columns (21 original and 5 new). All five new features are present, there are 0 missing values and 0 exact duplicate records. The raw dataset still has 10,194 rows and 21 columns, with none of the new columns, so it is unchanged.

Decision / Conclusion:

The working dataset passed validation and is ready to be saved.

### 6.2 Save Cleaned Dataset

Objective: Save the validated working dataset as a CSV file in the processed data folder for future business analysis.


#### 6.2.1 Export Cleaned Dataset

In [23]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_DATA_PATH = PROCESSED_DIR / 'cleaned_sales_data.csv'
clean_df.to_csv(CLEANED_DATA_PATH, index=False)

print("Dataset saved successfully")
print("Saved path:", CLEANED_DATA_PATH)

Dataset saved successfully
Saved path: c:\Users\pc\Desktop\Retail-Sales-and-Business-Analysis\data\processed\cleaned_sales_data.csv


#### 6.2.2 Verify Saved Dataset

In [24]:
saved_df = pd.read_csv(CLEANED_DATA_PATH)

print("File exists:", CLEANED_DATA_PATH.exists())
print("Saved dataset shape:", saved_df.shape)
print("Shape of the original working dataset:", clean_df.shape)
print("Columns match:", saved_df.columns.equals(clean_df.columns))


File exists: True
Saved dataset shape: (10194, 26)
Shape of the original working dataset: (10194, 26)
Columns match: True


Observations:

The processed dataset was saved as data/processed/cleaned_sales_data.csv. Reloading it returned 10,194 rows and 26 columns, and the column names match the working dataset.

Decision / Conclusion:

The processed dataset was saved and verified correctly, and is ready for the Day 3 analysis.

## 7. Day 2 Summary

Summarizing the data cleaning and feature engineering work completed in Day 2.

### 7.1 Key Findings

- Using November 15, 2026, as the reference date, 709 records (6.96%) have future Order Dates and 775 records (7.60%) have future Ship Dates.
- Future Order Dates range from November 16 to December 30, 2026, and future Ship Dates range from November 16, 2026, to January 5, 2027.
- 66 records were ordered on or before the reference date but shipped after it, which explains why future Ship Dates exceed future Order Dates.
- Missing values and exact duplicates were rechecked and none were found. No invalid quantity or discount values were found in the Day 1 investigation.

### 7.2 Cleaning Actions
- No records were deleted or modified.
- Future-dated records, negative profit records, statistical outliers and repeated Order IDs were all retained, because none of them were identified as errors.

### 7.3 Features Created

| Feature | Source | Purpose |
|---|---|---|
| Order Year | Order Date | Yearly analysis |
| Order Month | Order Date | Monthly analysis |
| Year-Month | Order Date | Monthly trends |
| Shipping Days | Ship Date - Order Date | Shipping performance |
| Profit Status | Profit | Profit/Loss classification |

### 7.4 Final Output

The validated dataset (10,194 rows and 26 columns) was saved as data/processed/cleaned_sales_data.csv. The raw dataset is unchanged.

### 7.5 Next Step

Day 3 will use the processed dataset for Pandas practice, business KPIs and product analysis.